In [2]:
import os

import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient
from qdrant_client import QdrantClient

load_dotenv()

mongo = MongoClient(os.getenv("MONGODB_URI", "mongodb://localhost:27017"))
qdrant = QdrantClient(url=os.getenv("QDRANT_URL", "http://192.168.1.7:6333"))

## MongoDB: Tasks

In [3]:
df_tasks = pd.DataFrame(list(mongo["patron_tasks"]["tasks"].find()))
print(f"Tasks: {len(df_tasks)} rows")
df_tasks


Tasks: 103 rows


,_id,user_id,chat_id,text,due_at,status,created_at,completed_at,special_instructions_for_agent,recurrence
0,b7a692f7-8ace-4418-a1cb-df0462521f19,283499750,283499750,Напомнить про terms of uses,2026-03-16 12:00:00,completed,2026-03-15 20:00:41.598,2026-03-16 12:00:43.696,NaN,NaN
1,a098e413-bd55-4355-90e5-a853734fbb37,351933633,351933633,Посмотреть с Сетти Лилихаймер,2026-03-15 22:51:00,completed,2026-03-15 21:51:46.117,2026-03-15 22:51:25.691,NaN,NaN
2,5344d942-3c91-411d-a0c9-49a1587e8ce4,351933633,351933633,Отправить деньги Максиму и оплатить свет,2026-03-15 23:22:00,completed,2026-03-15 21:52:23.345,2026-03-15 23:22:25.689,NaN,NaN
3,07008e1b-9bb4-4543-9192-c1349db22815,351933633,351933633,Про свет и Максима (отправить деньги Максиму и...,2026-03-17 10:00:00,completed,2026-03-16 00:08:14.256,2026-03-17 10:00:52.797,NaN,NaN
4,ef42b24a-95e1-4b3d-b58a-6e79540aad26,224713850,224713850,Ранкове повідомлення: надіслати план на день (...,2026-03-17 07:00:00,completed,2026-03-16 09:12:33.491,2026-03-17 07:00:52.796,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
98,96a81d03-fc68-4b6c-8dba-f68a5fa10f3b,327384331,327384331,Переглянути документацію по батерці,2026-03-24 19:40:00,completed,2026-03-24 17:10:58.122,2026-03-24 19:40:15.591,"Use a soft, gentle, and polite tone. No harsh ...",NaN
99,ec5c2c0d-f090-4075-94b4-946cc5547a3e,41481503,41481503,Пинг после ужина. Проверка готовности к налогам.,2026-03-24 23:30:00,completed,2026-03-24 22:38:06.573,2026-03-24 23:30:15.582,"Спроси, поел ли юзер и готов ли продолжить под...",NaN
100,03532b3b-d58e-49ec-9600-c5ac4c6efa4a,41481503,41481503,Окончание первого спринта подготовки документо...,2026-03-25 00:15:00,completed,2026-03-24 23:30:43.858,2026-03-25 00:15:15.586,"Сообщи юзеру, что 45 минут прошло. Пора сделат...",NaN
101,70a18cfb-67ec-491b-ab17-eb4d82f8f681,41481503,41481503,Продолжение работы над налогами (досортировка ...,2026-03-25 21:30:00,pending,2026-03-25 00:18:04.080,NaT,"Напомни юзеру, что пора возвращаться к налогам...",NaN


## MongoDB: Users

In [ ]:
df_users = pd.DataFrame(list(mongo["patron_users"]["users"].find()))
print(f"Users: {len(df_users)} rows")
df_users

## Qdrant: Memories

In [7]:
all_memories = []
offset = None
while True:
    result = qdrant.scroll(
        collection_name="memories",
        limit=100,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )
    points, next_offset = result
    for p in points:
        record = {"id": str(p.id), **p.payload}
        all_memories.append(record)
    if next_offset is None:
        break
    offset = next_offset

df_memories = pd.DataFrame(all_memories)
print(f"Memories: {len(df_memories)} rows")
df_memories.sort_values(by="created_at", ascending=False)

Memories: 73 rows


,id,user_id,text,created_at
52,9e8d9ae0-3cda-4b32-bc18-0613b4b1f483,351933633,Цель до субботы (28 марта 2026): отправить два...,2026-03-25T11:18:35.302756+00:00
5,18ea40a0-de5c-44e2-8622-58b159b3c272,351933633,25 марта 2026: Утренняя рутина OnlyFans выполн...,2026-03-25T10:40:33.066198+00:00
55,b028024a-4ee7-4281-99ea-e7c6112b6973,41481503,Пользователь подготовил онлайн место для налог...,2026-03-25T00:18:01.182761+00:00
1,0bf96d5b-3ce6-4766-89db-f52ed5edbfe0,351933633,"Пользователю не нравится фраза ""Serhii Setti e...",2026-03-24T15:10:28.133355+00:00
40,6ef3d793-587b-470d-ab58-84e889ce6401,351933633,24 марта 2026: Выполнена рутина по OnlyFans. П...,2026-03-24T14:57:38.591367+00:00
...,...,...,...,...
19,3ebc1d41-c5c2-40dd-acab-8baff2f8e8ef,224713850,Nutrition and Health Manual (70kg body weight)...,2026-03-16T09:10:42.870986+00:00
28,56adc396-6b52-450d-b3a3-f758a8a26363,224713850,"Hair care routine (Normal hair, mid-length, ma...",2026-03-16T09:10:03.905443+00:00
10,24ffe1c1-2a09-462b-a218-16674a22b715,283499750,"Пользователя зовут Станислав, ему 35 лет.",2026-03-15T16:45:29.565455+00:00
18,3a68d504-bd70-4d8b-bf56-a3ae4ab57414,176622453,"User likes data science subjects, mental healt...",2026-03-15T15:08:20.382053+00:00
